# 06b — Deploy the Chatbot App in **Pro** mode (OPTIONAL)

This notebook flips the already-deployed chatbot app (notebook `06`) into the richer
**pro** experience: a dockable workspace panel with an interactive knowledge graph, a
data explorer, an embedded Genie + AI/BI dashboard, and an animated multi-agent
"thinking" stream.

**Prerequisites**
1. Run `06_deploy_chatbot_app` first (creates the app + Lakebase).
2. Run `05b_create_ontobricks_graphrag_OPTIONAL` (creates `graphrag_vertices` / `graphrag_edges`).
3. A SQL warehouse the app can use (auto-discovered, or set `sql_warehouse_id` in `config.py`).
4. (Optional) A published Genie space and AI/BI dashboard to embed.

This notebook is **idempotent** and only rewrites the app's environment + permissions.
It does not touch the simple-mode deployment if you skip it.

In [ ]:
# Load shared project config (catalog, schema, app names, and the new pro-mode vars)
exec(open('../config.py').read())
print(f"app_mode        = {app_mode}")
print(f"catalog/schema  = {catalog}.{schema}")
print(f"graph tables in = {graph_catalog}.{graph_schema}")
print(f"app             = {app_name}-{app_resource_suffix}")

In [ ]:
dbutils.widgets.text("host", "", "Workspace host (e.g. dbc-xxxx.cloud.databricks.com)")
host = dbutils.widgets.get("host")

In [ ]:
import os, sys, json, time, base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.runtime import dbutils

notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
if host:
    os.environ["DATABRICKS_HOST"] = f"https://{host}"

w = WorkspaceClient()

# App name handling mirrors notebook 06 (30-char limit).
app_full_name = f"{app_name}-{app_resource_suffix}"[:30].rstrip("-")
print("Target app:", app_full_name)

## Step 1 — Verify the pro-mode data is present

The Graph and Data Explorer tabs read three tables via the SQL warehouse. If any are
missing, run the relevant setup notebook before continuing.

In [ ]:
required = ["graphrag_vertices", "graphrag_edges", "ticker_data"]
missing = []
for t in required:
    try:
        spark.sql(f"SELECT 1 FROM {graph_catalog}.{graph_schema}.{t} LIMIT 1")
    except Exception:
        missing.append(t)

if missing:
    print("⚠️  Missing tables:", missing)
    print("    -> run 00_tables_setup and 05b_create_ontobricks_graphrag_OPTIONAL first.")
else:
    print("✅ graphrag_vertices, graphrag_edges, ticker_data all present")

## Step 2 — Resolve the SQL warehouse

Uses `sql_warehouse_id` from `config.py` if set, otherwise auto-selects an available
warehouse (preferring serverless).

In [ ]:
warehouse_id = (sql_warehouse_id or "").strip()
if not warehouse_id:
    whs = list(w.warehouses.list())
    serverless = [x for x in whs if getattr(x, "enable_serverless_compute", False)]
    pick = (serverless or whs or [None])[0]
    warehouse_id = getattr(pick, "id", "") or ""
    print("Auto-selected warehouse:", getattr(pick, "name", None), warehouse_id)

assert warehouse_id, "No SQL warehouse found. Create one or set sql_warehouse_id in config.py."
print("Warehouse id:", warehouse_id)

## Step 3 — Resolve the Genie space + AI/BI dashboard to embed (optional)

Set `genie_space_id` / `aibi_dashboard_id` (or the explicit `*_embed_url`) in `config.py`
for full control. Otherwise we make a best-effort discovery of an AI/BI dashboard by name.
Leaving these blank simply hides the Dashboard tab.

In [ ]:
genie_id  = (genie_space_id or "").strip()
genie_url = (genie_embed_url or "").strip()
# Discover the Genie space by title if no id was pinned in config.py.
if not genie_id:
    try:
        spaces = w.api_client.do("GET", "/api/2.0/genie/spaces").get("spaces", [])
        match = next((sp for sp in spaces if (sp.get("title") or "").strip() == "Financial_Data_Explorer"), None)
        if match:
            genie_id = match.get("space_id", "")
            print("Discovered Genie space 'Financial_Data_Explorer':", genie_id)
    except Exception as e:
        print("Genie discovery skipped:", e)

dash_id   = (aibi_dashboard_id or "").strip()
dash_url  = (aibi_embed_url or "").strip()

# Best-effort: find an AI/BI (Lakeview) dashboard whose name mentions mag7.
if not dash_id and not dash_url:
    try:
        for d in w.lakeview.list():
            name = (getattr(d, "display_name", "") or "").lower()
            if "mag7" in name or "mag 7" in name:
                dash_id = d.dashboard_id
                print("Found AI/BI dashboard:", d.display_name, "->", dash_id)
                break
    except Exception as e:
        print("Dashboard auto-discovery skipped:", e)

print("genie_space_id   =", genie_id or "(none)")
print("aibi_dashboard_id=", dash_id or "(none)")

## Step 4 — Write the pro environment into `app.yaml`

Databricks Apps read environment variables from `app.yaml` in the deployed source. We
rewrite that file in the workspace copy of the repo with the resolved values, then
redeploy in Step 6.

In [ ]:
repo_root = os.path.dirname(os.path.dirname(notebook_context.notebookPath().get()))
app_src = f"/Workspace{repo_root}/chatbot-app"

app_yaml = f'''command: ["npm", "run", "start"]
runtime: nodejs20

env:
  - name: DATABRICKS_SERVING_ENDPOINT
    valueFrom: serving-endpoint
  - name: APP_MODE
    value: "pro"
  - name: DATABRICKS_WAREHOUSE_ID
    value: "{warehouse_id}"
  - name: GRAPH_CATALOG
    value: "{graph_catalog}"
  - name: GRAPH_SCHEMA
    value: "{graph_schema}"
  - name: GENIE_SPACE_ID
    value: "{genie_id}"
  - name: GENIE_EMBED_URL
    value: "{genie_url}"
  - name: AIBI_DASHBOARD_ID
    value: "{dash_id}"
  - name: AIBI_EMBED_URL
    value: "{dash_url}"
'''

w.api_client.do("POST", "/api/2.0/workspace/import", body={
    "path": f"{app_src}/app.yaml",
    "format": "AUTO",
    "overwrite": True,
    "content": base64.b64encode(app_yaml.encode()).decode(),
})
print("✅ Wrote pro app.yaml to", f"{app_src}/app.yaml")
print(app_yaml)

## Step 5 — Grant the app service principal access

The app queries the warehouse and reads the graph/ticker tables as its service
principal, so it needs `CAN_USE` on the warehouse and `SELECT` on the tables.

In [ ]:
app = w.api_client.do("GET", f"/api/2.0/apps/{app_full_name}")
# Field names can vary across workspaces; try the common ones.
sp_name = (
    app.get("service_principal_name")
    or app.get("service_principal_client_id")
    or app.get("service_principal_id")
)
print("App service principal:", sp_name)

# Warehouse CAN_USE
try:
    w.api_client.do(
        "PATCH",
        f"/api/2.0/permissions/warehouses/{warehouse_id}",
        body={"access_control_list": [
            {"service_principal_name": str(sp_name), "permission_level": "CAN_USE"}
        ]},
    )
    print("✅ Granted CAN_USE on warehouse")
except Exception as e:
    print("⚠️  Warehouse grant issue (grant manually if needed):", e)

# Catalog / schema / table SELECT
grants = [
    (f"USE CATALOG", f"CATALOG `{graph_catalog}`"),
    (f"USE SCHEMA", f"SCHEMA `{graph_catalog}`.`{graph_schema}`"),
]
for priv, obj in grants:
    try:
        spark.sql(f"GRANT {priv} ON {obj} TO `{sp_name}`")
    except Exception as e:
        print(f"⚠️  grant {priv} on {obj}:", e)

for t in ["graphrag_vertices", "graphrag_edges", "ticker_data"]:
    try:
        spark.sql(
            f"GRANT SELECT ON TABLE `{graph_catalog}`.`{graph_schema}`.`{t}` TO `{sp_name}`"
        )
    except Exception as e:
        print(f"⚠️  grant SELECT on {t}:", e)
print("Permission grants attempted.")

## Step 6 — Redeploy the app with the pro configuration

In [ ]:
deploy = w.api_client.do(
    "POST",
    f"/api/2.0/apps/{app_full_name}/deployments",
    body={"source_code_path": app_src},
)
print("Triggered deployment:", deploy.get("deployment_id"))
print("\nWatch progress under Workspace > Apps >", app_full_name)

## 🎉 Pro mode enabled

**Embedding note:** the Dashboard tab loads the Genie space and AI/BI dashboard in
iframes. Make sure both are **published** and that embedding from the app domain is
allowed (AI/BI dashboards expose a published *Embed* URL — paste it into
`aibi_embed_url` in `config.py` for the most reliable result; same for `genie_embed_url`).

To revert to the simple app, re-run `06_deploy_chatbot_app` (which writes the simple
`app.yaml`) or set `app_mode = "simple"` and re-run this notebook.